In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS main.silver;

CREATE OR REPLACE TABLE main.silver.customer_master (
    customer_id INT,
    customer_name STRING,
    city STRING,
    balance DECIMAL(10,2)
);

In [0]:
%sql

INSERT INTO main.silver.customer_master VALUES
(101,'Rajesh Kumar','Hyderabad',45000),
(102,'Priya Sharma','Bengaluru',68000),
(103,'Arjun Reddy','Chennai',52000),
(104,'Sneha Iyer','Mumbai',91000);

In [0]:
%sql

SELECT *
FROM main.silver.customer_master
ORDER BY customer_id;

# Creating staging table to heve daily updates

In [0]:
%sql

CREATE OR REPLACE TABLE main.bronze.customer_updates (
    customer_id INT,
    customer_name STRING,
    city STRING,
    balance DECIMAL(10,2)
);

INSERT INTO main.bronze.customer_updates VALUES
(101,'Rajesh Kumar','Hyderabad',55000),
(103,'Arjun Reddy','Pune',52000),
(105,'Ananya Gupta','Delhi',78000),
(106,'Vikram Singh','Kolkata',64000);

In [0]:
%sql

SELECT *
FROM main.bronze.customer_updates
ORDER BY customer_id;

# Merge

In [0]:
%sql

MERGE INTO main.silver.customer_master AS target
USING main.bronze.customer_updates AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.balance = source.balance

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    city,
    balance
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.balance
);

In [0]:
%sql

SELECT *
FROM main.silver.customer_master
ORDER BY customer_id;

# Let's see how Delta Lake recorded this MERGE

In [0]:
%sql

DESCRIBE HISTORY main.silver.customer_master;